In [1]:
!pip install torchaudio librosa transformers datasets matplotlib soundfile pandas tqdm scipy pesq jiwe

  Preparing metadata (setup.py) ... done
ERROR: Could not find a version that satisfies the requirement jiwe (from versions: none)
ERROR: No matching distribution found for jiwe


In [2]:
# SETUP CELL - Run First
from google.colab import drive
import os
import warnings
warnings.filterwarnings('ignore')

# Mount Drive
drive.mount('/content/drive')

# Project Directory
PROJECT_PATH = "/content/drive/MyDrive/q1_cepstral_pipeline"
os.makedirs(f"{PROJECT_PATH}/outputs", exist_ok=True)
os.makedirs(f"{PROJECT_PATH}/data", exist_ok=True)

# Install Dependencies
!pip install numpy scipy matplotlib torch torchaudio transformers jiwer librosa --quiet

# Clean Cache
!rm -rf ~/.cache/pip

print(f" Project Path: {PROJECT_PATH}")
print(f" Python: {__import__('sys').version.split()[0]}")

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 31.2 MB/s eta 0:00:00
 Project Path: /content/drive/MyDrive/q1_cepstral_pipeline
 Python: 3.12.13


In [3]:
 %%writefile $PROJECT_PATH/mfcc_manual.py
"""
Manual MFCC/Cepstrum Feature Extraction Engine
Supports: Synthetic signals AND real audio files (.wav, .m4a, .flac, .mp3)
"""

import numpy as np
from scipy.fftpack import dct, fft
from scipy.signal import get_window
import matplotlib.pyplot as plt
import os
import sys

# Default project path
DEFAULT_PROJECT_PATH = "/content/drive/MyDrive/q1_cepstral_pipeline"

def load_audio_file(filepath, target_sr=16000):
    """
    Load audio file with automatic resampling

    Args:
        filepath: Path to audio file (.wav, .m4a, .flac, .mp3)
        target_sr: Target sample rate (default: 16000)

    Returns:
        signal: Mono audio as numpy array
        sample_rate: Actual sample rate after resampling
    """
    try:
        import torchaudio

        # Load with torchaudio (supports M4A via ffmpeg)
        waveform, sr = torchaudio.load(filepath)

        # Convert to mono if stereo
        if waveform.shape[0] > 1:
            waveform = waveform.mean(dim=0, keepdim=True)

        # Resample if needed
        if sr != target_sr:
            waveform = torchaudio.functional.resample(waveform, sr, target_sr)
            sr = target_sr

        # Convert to numpy and normalize
        signal = waveform.numpy().flatten()
        signal = signal / (np.max(np.abs(signal)) + 1e-8)  # Normalize to [-1, 1]

        return signal, sr

    except ImportError:
        # Fallback to librosa
        import librosa
        signal, sr = librosa.load(filepath, sr=target_sr, mono=True)
        signal = signal / (np.max(np.abs(signal)) + 1e-8)
        return signal, sr

    except Exception as e:
        print(f" Error loading {filepath}: {e}")
        return None, None


class ManualMFCC:
    """Handcrafted MFCC extraction from first principles"""

    def __init__(self, sample_rate=16000, n_mfcc=13, n_fft=512,
                 hop_length=160, win_length=400, n_mels=26):
        self.sample_rate = sample_rate
        self.n_mfcc = n_mfcc
        self.n_fft = n_fft
        self.hop_length = hop_length
        self.win_length = win_length
        self.n_mels = n_mels
        self.preemphasis = 0.97
        self.mel_filters = self._create_mel_filterbank()

    def _pre_emphasis(self, signal):
        """Apply pre-emphasis filter: y[t] = x[t] - α·x[t-1]"""
        return np.append(signal[0], signal[1:] - self.preemphasis * signal[:-1])

    def _frame_signal(self, signal):
        """Frame signal with overlap"""
        num_frames = 1 + int(np.ceil((len(signal) - self.win_length) / self.hop_length))
        frames = np.zeros((num_frames, self.win_length))
        for i in range(num_frames):
            start = i * self.hop_length
            end = min(start + self.win_length, len(signal))
            frames[i, :end-start] = signal[start:end]
        return frames

    def _create_mel_filterbank(self):
        """Create triangular Mel-scale filterbank"""
        freqs = np.fft.rfftfreq(self.n_fft, 1/self.sample_rate)
        def hz_to_mel(hz): return 2595 * np.log10(1 + hz/700)
        def mel_to_hz(mel): return 700 * (10**(mel/2595) - 1)
        mel_min = hz_to_mel(0)
        mel_max = hz_to_mel(self.sample_rate/2)
        mel_points = np.linspace(mel_min, mel_max, self.n_mels + 2)
        hz_points = mel_to_hz(mel_points)
        filters = np.zeros((self.n_mels, len(freqs)))
        for m in range(1, self.n_mels + 1):
            f_lower = hz_points[m-1]
            f_center = hz_points[m]
            f_upper = hz_points[m+1]
            for i, f in enumerate(freqs):
                if f_lower <= f < f_center:
                    filters[m-1, i] = (f - f_lower) / (f_center - f_lower)
                elif f_center <= f < f_upper:
                    filters[m-1, i] = (f_upper - f) / (f_upper - f_center)
        return filters

    def extract_mfcc(self, signal, window_type='hamming'):
        """Extract MFCC features manually"""
        signal = self._pre_emphasis(signal.astype(np.float64))
        frames = self._frame_signal(signal)
        window = get_window(window_type, self.win_length)
        frames = frames * window
        fft_frames = np.abs(fft(frames, n=self.n_fft))[:, :self.n_fft//2 + 1]
        power_spectrum = fft_frames ** 2
        mel_spectrum = np.dot(power_spectrum, self.mel_filters.T)
        mel_spectrum = np.where(mel_spectrum == 0, np.finfo(float).eps, mel_spectrum)
        log_mel = np.log(mel_spectrum)
        mfcc = dct(log_mel, type=2, axis=1, norm='ortho')[:, :self.n_mfcc]
        return mfcc

    def extract_cepstrum(self, signal, window_type='hamming'):
        """Extract cepstrum for quefrency analysis"""
        signal = self._pre_emphasis(signal.astype(np.float64))
        frames = self._frame_signal(signal)
        window = get_window(window_type, self.win_length)
        frames = frames * window
        cepstra = []
        for frame in frames:
            spectrum = np.abs(fft(frame, n=self.n_fft))[:self.n_fft//2 + 1]
            spectrum = np.where(spectrum == 0, np.finfo(float).eps, spectrum)
            log_spectrum = np.log(spectrum)
            cepstrum = np.real(fft(log_spectrum))
            cepstra.append(cepstrum)
        cepstra = np.array(cepstra)
        quefrency_split = int(0.004 * self.sample_rate)
        low_quefrency = cepstra[:, :quefrency_split]
        high_quefrency = cepstra[:, quefrency_split:quefrency_split*3]
        return low_quefrency, high_quefrency, cepstra


def plot_mfcc_comparison(signal, sample_rate=16000, project_path=None, title="MFCC Analysis"):
    """Visualize MFCC extraction stages"""
    if project_path is None:
        project_path = DEFAULT_PROJECT_PATH

    mfcc_engine = ManualMFCC(sample_rate=sample_rate)
    mfcc = mfcc_engine.extract_mfcc(signal)
    low_q, high_q, cepstra = mfcc_engine.extract_cepstrum(signal)

    fig, axes = plt.subplots(3, 2, figsize=(14, 10))

    # Original signal (first 2000 samples)
    axes[0, 0].plot(signal[:2000])
    axes[0, 0].set_title('Original Signal (first 2000 samples)')
    axes[0, 0].set_xlabel('Samples')
    axes[0, 0].set_ylabel('Amplitude')
    axes[0, 0].grid(True, alpha=0.3)

    # Spectrogram
    from scipy.signal import spectrogram
    f, t_spec, Sxx = spectrogram(signal, fs=sample_rate, nperseg=256)
    axes[0, 1].pcolormesh(t_spec, f, 10*np.log10(Sxx + 1e-10), shading='gouraud')
    axes[0, 1].set_title('Spectrogram')
    axes[0, 1].set_xlabel('Time (s)')
    axes[0, 1].set_ylabel('Frequency (Hz)')

    # MFCC heatmap
    axes[1, 0].imshow(mfcc.T, aspect='auto', origin='lower', cmap='viridis')
    axes[1, 0].set_title(f'MFCC Coefficients ({mfcc.shape[1]} × {mfcc.shape[0]})')
    axes[1, 0].set_xlabel('Frame')
    axes[1, 0].set_ylabel('Coefficient')

    # Cepstrum
    axes[1, 1].plot(np.mean(cepstra, axis=0)[:200])
    axes[1, 1].axvline(x=64, color='red', linestyle='--', label='4ms split')
    axes[1, 1].set_title('Mean Cepstrum (Quefrency Domain)')
    axes[1, 1].set_xlabel('Quefrency (samples)')
    axes[1, 1].set_ylabel('Magnitude')
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)

    # Low vs High Quefrency
    axes[2, 0].plot(np.mean(low_q, axis=0), label='Low Quefrency (<4ms)', color='blue')
    axes[2, 0].set_title('Low Quefrency: Vocal Tract Envelope')
    axes[2, 0].set_xlabel('Quefrency')
    axes[2, 0].set_ylabel('Magnitude')
    axes[2, 0].legend()
    axes[2, 0].grid(True, alpha=0.3)

    axes[2, 1].plot(np.mean(high_q, axis=0), label='High Quefrency (>4ms)', color='red')
    axes[2, 1].set_title('High Quefrency: Pitch Periodicity')
    axes[2, 1].set_xlabel('Quefrency')
    axes[2, 1].set_ylabel('Magnitude')
    axes[2, 1].legend()
    axes[2, 1].grid(True, alpha=0.3)

    plt.suptitle(title, fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()

    output_path = f"{project_path}/outputs/mfcc_plots.pdf"
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    plt.close()

    print(f" MFCC plots saved to {output_path}")
    return mfcc, cepstra


def run_demo_synthetic(project_path=None):
    """Run demo with synthetic signal"""
    if project_path is None:
        project_path = DEFAULT_PROJECT_PATH

    print("🔧 Testing Manual MFCC Engine with Synthetic Signal...")

    sample_rate = 16000
    duration = 2.0
    t = np.linspace(0, duration, int(sample_rate * duration))

    # Voiced: harmonic signal
    voiced = 0.5 * np.sin(2*np.pi*150*t) + 0.3*np.sin(2*np.pi*300*t)
    # Unvoiced: noise-like
    unvoiced = 0.2 * np.random.randn(len(t))

    # Combine
    signal = np.zeros_like(t)
    signal[:len(t)//2] = voiced[:len(t)//2]
    signal[len(t)//2:] = unvoiced[len(t)//2]

    mfcc, cepstra = plot_mfcc_comparison(signal, sample_rate, project_path,
                                         title="Synthetic Signal: Voiced + Unvoiced")

    print(f" MFCC shape: {mfcc.shape}")
    print(f" Cepstrum shape: {cepstra.shape}")
    print(" Manual MFCC engine working!")

    return mfcc, cepstra


def run_demo_real(filepath, project_path=None):
    """Run analysis on real audio file"""
    if project_path is None:
        project_path = DEFAULT_PROJECT_PATH

    print(f" Loading real audio: {filepath}")

    # Load audio
    signal, sample_rate = load_audio_file(filepath, target_sr=16000)

    if signal is None:
        print(f" Failed to load {filepath}")
        return None, None

    print(f" Loaded: {len(signal)/sample_rate:.2f} seconds @ {sample_rate} Hz")

    # Extract and plot
    filename = os.path.basename(filepath)
    title = f"Real Speech: {filename}"

    mfcc, cepstra = plot_mfcc_comparison(signal, sample_rate, project_path, title=title)

    print(f" MFCC shape: {mfcc.shape}")
    print(f" Cepstrum shape: {cepstra.shape}")
    print(f" Duration: {len(signal)/sample_rate:.2f}s")
    print(f" Frames: {mfcc.shape[0]} (10ms hop)")

    # Save processed signal for other modules
    output_signal = f"{project_path}/data/processed_signal.npy"
    np.save(output_signal, {'signal': signal, 'sample_rate': sample_rate})
    print(f" Processed signal saved to {output_signal}")

    return mfcc, cepstra


if __name__ == "__main__":
    import argparse

    parser = argparse.ArgumentParser(description='Manual MFCC/Cepstrum Engine')
    parser.add_argument('--input', '-i', type=str, default=None,
                       help='Path to real audio file (.wav, .m4a, .flac, .mp3)')
    parser.add_argument('--project-path', '-p', type=str, default=DEFAULT_PROJECT_PATH,
                       help='Project output directory')
    parser.add_argument('--synthetic', '-s', action='store_true',
                       help='Use synthetic demo signal (default)')

    args = parser.parse_args()
    project_path = args.project_path

    # Ensure output directory exists
    os.makedirs(f"{project_path}/outputs", exist_ok=True)

    if args.input and os.path.exists(args.input):
        # Process real audio file
        run_demo_real(args.input, project_path)
    else:
        # Fall back to synthetic demo
        print("  No valid input file provided, using synthetic demo")
        run_demo_synthetic(project_path)

Overwriting /content/drive/MyDrive/q1_cepstral_pipeline/mfcc_manual.py


In [4]:
#  TEST WITH YOUR REAL M4A FILE
PROJECT_PATH = "/content/drive/MyDrive/q1_cepstral_pipeline"
AUDIO_FILE = "/content/drive/MyDrive/q1_cepstral_pipeline/data/voice_speech_Q1_1.m4a"

# Verify file exists
import os
if os.path.exists(AUDIO_FILE):
    print(f" Found audio file: {AUDIO_FILE}")
    print(f" File size: {os.path.getsize(AUDIO_FILE) / 1024:.1f} KB")
else:
    print(f" File not found: {AUDIO_FILE}")
    print(" Make sure the file is uploaded to Google Drive")

# Run MFCC analysis on real file
%run $PROJECT_PATH/mfcc_manual.py --input "$AUDIO_FILE" --project-path "$PROJECT_PATH"

 Found audio file: /content/drive/MyDrive/q1_cepstral_pipeline/data/voice_speech_Q1_1.m4a
 File size: 815.0 KB
 Loading real audio: /content/drive/MyDrive/q1_cepstral_pipeline/data/voice_speech_Q1_1.m4a
 Loaded: 32.64 seconds @ 16000 Hz
 MFCC plots saved to /content/drive/MyDrive/q1_cepstral_pipeline/outputs/mfcc_plots.pdf
 MFCC shape: (3263, 13)
 Cepstrum shape: (3263, 257)
 Duration: 32.64s
 Frames: 3263 (10ms hop)
 Processed signal saved to /content/drive/MyDrive/q1_cepstral_pipeline/data/processed_signal.npy


<Figure size 640x480 with 0 Axes>

In [8]:
%%writefile $PROJECT_PATH/leakage_snr.py
"""
Spectral Leakage and SNR Analysis for Different Window Functions
Supports: Synthetic tones AND real speech audio files
"""

import numpy as np
from scipy.signal import get_window
from scipy.fftpack import fft
import matplotlib.pyplot as plt
import os
import sys

DEFAULT_PROJECT_PATH = "/content/drive/MyDrive/q1_cepstral_pipeline"

def load_audio_for_analysis(filepath, target_sr=16000, max_duration=0.1):
    """
    Load and prepare audio for spectral analysis

    Args:
        filepath: Path to audio file
        target_sr: Target sample rate
        max_duration: Max duration in seconds for FFT analysis (default: 100ms)

    Returns:
        signal: Normalized audio segment for analysis
        sample_rate: Actual sample rate
    """
    try:
        import torchaudio
        waveform, sr = torchaudio.load(filepath)
        if waveform.shape[0] > 1:
            waveform = waveform.mean(dim=0, keepdim=True)
        if sr != target_sr:
            waveform = torchaudio.functional.resample(waveform, sr, target_sr)
            sr = target_sr
        signal = waveform.numpy().flatten()
        signal = signal / (np.max(np.abs(signal)) + 1e-8)

        # Take a short segment for spectral analysis (avoid long signals)
        max_samples = int(max_duration * sr)
        if len(signal) > max_samples:
            # Find a voiced segment (higher energy) for better analysis
            energy = np.convolve(signal**2, np.ones(int(0.01*sr))/int(0.01*sr), mode='valid')
            start_idx = np.argmax(energy) if len(energy) > 0 else 0
            signal = signal[start_idx:start_idx+max_samples]

        return signal, sr
    except Exception as e:
        print(f"  Error loading {filepath}: {e}")
        return None, None


def calculate_spectral_leakage(signal, sample_rate=16000, window_type='hamming'):
    """Measure spectral leakage using main-lobe to side-lobe ratio"""
    window = get_window(window_type, len(signal))
    windowed = signal * window
    N = len(windowed)
    freqs = np.fft.rfftfreq(N, 1/sample_rate)
    spectrum = np.abs(fft(windowed))[:N//2 + 1]
    spectrum_db = 20 * np.log10(spectrum + 1e-10)

    main_peak_idx = np.argmax(spectrum)
    main_peak_mag = spectrum_db[main_peak_idx]

    # -3dB main lobe width
    threshold = main_peak_mag - 3
    left_idx, right_idx = main_peak_idx, main_peak_idx
    while left_idx > 0 and spectrum_db[left_idx] > threshold:
        left_idx -= 1
    while right_idx < len(spectrum_db)-1 and spectrum_db[right_idx] > threshold:
        right_idx += 1
    main_lobe_width = freqs[right_idx] - freqs[left_idx]

    # Side-lobe level
    side_lobe_mask = np.ones(len(spectrum_db), dtype=bool)
    side_lobe_mask[left_idx:right_idx] = False
    side_lobe_level = np.mean(spectrum_db[side_lobe_mask])

    leakage_ratio = main_peak_mag - side_lobe_level

    return {
        'leakage_ratio': leakage_ratio,
        'main_lobe_width': main_lobe_width,
        'side_lobe_level': side_lobe_level
    }


def calculate_snr_proxy(signal, sample_rate=16000, window_type='hamming'):
    """Estimate SNR using spectral flatness"""
    window = get_window(window_type, len(signal))
    windowed = signal * window
    spectrum = np.abs(fft(windowed))[:len(windowed)//2 + 1]
    spectrum = np.where(spectrum == 0, 1e-10, spectrum)

    geom_mean = np.exp(np.mean(np.log(spectrum)))
    arith_mean = np.mean(spectrum)
    spectral_flatness = geom_mean / arith_mean
    snr_db = -10 * np.log10(spectral_flatness + 1e-10)

    return {'snr_db': snr_db, 'spectral_flatness': spectral_flatness}


def compare_windows(signal, sample_rate=16000):
    """Compare all three window functions"""
    windows = ['boxcar', 'hamming', 'hann']
    results = {}
    for w in windows:
        leakage = calculate_spectral_leakage(signal, sample_rate, w)
        snr = calculate_snr_proxy(signal, sample_rate, w)
        results[w] = {**leakage, **snr}
    return results


def plot_window_comparison(signal, sample_rate=16000, project_path=None, title="Window Comparison"):
    """Generate comparison plots and table"""
    if project_path is None:
        project_path = DEFAULT_PROJECT_PATH

    results = compare_windows(signal, sample_rate)
    display_names = {'boxcar': 'Rectangular', 'hamming': 'Hamming', 'hann': 'Hanning'}

    print("\n" + "="*70)
    print(" SPECTRAL LEAKAGE & SNR COMPARISON")
    print("="*70)
    print(f"{'Window':<15} {'Leakage Ratio':>15} {'Main Lobe (Hz)':>18} {'SNR (dB)':>12}")
    print("-"*70)
    for w, r in results.items():
        name = display_names.get(w, w)
        print(f"{name:<15} {r['leakage_ratio']:>15.2f} {r['main_lobe_width']:>18.2f} {r['snr_db']:>12.2f}")
    print("="*70)

    # Plot
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    N = 512
    t = np.arange(N)

    for idx, w_type in enumerate(['boxcar', 'hamming', 'hann']):
        ax = axes[idx//2, idx%2]
        window = get_window(w_type, N)
        ax.plot(t, window, linewidth=2)
        ax.set_title(f'{display_names[w_type]} Window')
        ax.set_xlabel('Sample')
        ax.set_ylabel('Amplitude')
        ax.grid(True, alpha=0.3)

    # Spectral response with 1000 Hz tone
    ax = axes[1, 1]
    freqs = np.fft.rfftfreq(512, 1/sample_rate)
    test_signal = np.sin(2*np.pi*1000*np.arange(512)/sample_rate)
    for w_type in ['boxcar', 'hamming', 'hann']:
        window = get_window(w_type, 512)
        spectrum = np.abs(fft(test_signal * window))[:257]
        spectrum_db = 20*np.log10(spectrum + 1e-10)
        ax.plot(freqs, spectrum_db, label=display_names[w_type], linewidth=1.5)
    ax.set_title('Spectral Response (1000 Hz Tone)')
    ax.set_xlabel('Frequency (Hz)')
    ax.set_ylabel('Magnitude (dB)')
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.set_xlim(0, 4000)

    plt.suptitle(title, fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig(f"{project_path}/outputs/leakage_comparison.pdf", dpi=300, bbox_inches='tight')
    plt.close()

    print(f"\n Comparison plot saved to {project_path}/outputs/leakage_comparison.pdf")

    # Save results
    import json
    results_display = {display_names.get(k, k): v for k, v in results.items()}
    with open(f"{project_path}/outputs/leakage_results.json", 'w') as f:
        json.dump(results_display, f, indent=2)

    return results


def run_demo_synthetic(project_path=None):
    """Run demo with synthetic 1000 Hz tone"""
    if project_path is None:
        project_path = DEFAULT_PROJECT_PATH

    print(" Analyzing Spectral Leakage & SNR (Synthetic Tone)...")
    sample_rate = 16000
    duration = 0.1
    t = np.linspace(0, duration, int(sample_rate * duration))
    signal = np.sin(2*np.pi*1000*t) + 0.1*np.random.randn(len(t))

    results = plot_window_comparison(signal, sample_rate, project_path,
                                     title="Spectral Analysis: 1000 Hz Synthetic Tone")

    best_window = max(results.keys(), key=lambda w: results[w]['leakage_ratio'])
    display_names = {'boxcar': 'Rectangular', 'hamming': 'Hamming', 'hann': 'Hanning'}
    print(f"\n Recommendation: Use '{display_names[best_window]}' window for best leakage suppression")

    return results


def run_demo_real(filepath, project_path=None):
    """Run spectral analysis on real speech audio"""
    if project_path is None:
        project_path = DEFAULT_PROJECT_PATH

    print(f" Analyzing Spectral Leakage & SNR for: {os.path.basename(filepath)}")

    signal, sample_rate = load_audio_for_analysis(filepath, target_sr=16000, max_duration=0.1)
    if signal is None:
        print(" Failed to load audio for analysis")
        return None

    print(f" Analyzing {len(signal)/sample_rate*1000:.1f}ms segment @ {sample_rate} Hz")

    filename = os.path.basename(filepath)
    results = plot_window_comparison(signal, sample_rate, project_path,
                                     title=f"Spectral Analysis: {filename}")

    best_window = max(results.keys(), key=lambda w: results[w]['leakage_ratio'])
    display_names = {'boxcar': 'Rectangular', 'hamming': 'Hamming', 'hann': 'Hanning'}
    print(f"\n For this speech segment: '{display_names[best_window]}' window recommended")

    return results


if __name__ == "__main__":
    import argparse
    parser = argparse.ArgumentParser(description='Spectral Leakage & SNR Analysis')
    parser.add_argument('--input', '-i', type=str, default=None, help='Path to real audio file')
    parser.add_argument('--project-path', '-p', type=str, default=DEFAULT_PROJECT_PATH)
    parser.add_argument('--synthetic', '-s', action='store_true', help='Use synthetic tone')
    args = parser.parse_args()

    os.makedirs(f"{args.project_path}/outputs", exist_ok=True)

    if args.input and os.path.exists(args.input):
        run_demo_real(args.input, args.project_path)
    else:
        print("  No valid input file, using synthetic 1000 Hz tone")
        run_demo_synthetic(args.project_path)

Overwriting /content/drive/MyDrive/q1_cepstral_pipeline/leakage_snr.py


In [11]:
%%writefile $PROJECT_PATH/voiced_unvoiced.py
"""
Voiced/Unvoiced Boundary Detection using Cepstrum Analysis
Uses low-quefrency (vocal tract) and high-quefrency (pitch) regions
FIXED: Consistent frame parameters across all features
"""

import numpy as np
from scipy.signal import find_peaks
from mfcc_manual import ManualMFCC
import matplotlib.pyplot as plt

class VoicedUnvoicedDetector:
    """Detect voiced/unvoiced segments using cepstral features"""

    def __init__(self, sample_rate=16000, frame_length=400, hop_length=160, quefrency_split_ms=4):
        """
        Args:
            sample_rate: Audio sample rate (Hz)
            frame_length: Frame size in samples (25ms @ 16kHz = 400)
            hop_length: Hop size in samples (10ms @ 16kHz = 160)
            quefrency_split_ms: Quefrency split point in milliseconds
        """
        self.sample_rate = sample_rate
        self.frame_length = frame_length
        self.hop_length = hop_length
        self.quefrency_split = int(quefrency_split_ms * sample_rate / 1000)
        self.mfcc_engine = ManualMFCC(
            sample_rate=sample_rate,
            win_length=frame_length,
            hop_length=hop_length
        )

    def _compute_zcr(self, signal):
        """Zero-crossing rate per frame (consistent with cepstrum frames)"""
        zcr = []
        for i in range(0, len(signal) - self.frame_length, self.hop_length):
            frame = signal[i:i+self.frame_length]
            zc = np.sum(np.abs(np.diff(np.signbit(frame)))) / (2 * len(frame))
            zcr.append(zc)
        return np.array(zcr)

    def _compute_energy(self, signal):
        """Frame energy (consistent with cepstrum frames)"""
        energy = []
        for i in range(0, len(signal) - self.frame_length, self.hop_length):
            frame = signal[i:i+self.frame_length]
            energy.append(np.sum(frame**2))
        return np.array(energy)

    def _compute_pitch_strength(self, cepstra):
        """Measure pitch periodicity from high-quefrency cepstrum"""
        # Pitch typically appears at quefrency 2-20ms
        pitch_region = cepstra[:, self.quefrency_split:min(self.quefrency_split*5, cepstra.shape[1])]

        # Energy in pitch region as pitch strength proxy
        pitch_strength = np.sum(np.abs(pitch_region), axis=1)

        return pitch_strength

    def _compute_vocal_tract_energy(self, low_quefrency):
        """Energy in low-quefrency region (vocal tract envelope)"""
        return np.sum(np.abs(low_quefrency), axis=1)

    def detect_boundaries(self, signal):
        """
        Detect voiced/unvoiced boundaries

        Returns:
            boundaries: list of (time, label) tuples
            labels: array of 'voiced'/'unvoiced' per frame
            voiced_score: confidence score per frame
            frame_times: time position of each frame
        """
        # Extract cepstral features
        low_q, high_q, cepstra = self.mfcc_engine.extract_cepstrum(signal)

        # Compute ALL features with consistent framing
        zcr = self._compute_zcr(signal)
        energy = self._compute_energy(signal)
        pitch_strength = self._compute_pitch_strength(cepstra)
        vt_energy = self._compute_vocal_tract_energy(low_q)

        # Debug: Check shapes match
        min_len = min(len(zcr), len(energy), len(pitch_strength), len(vt_energy))
        zcr = zcr[:min_len]
        energy = energy[:min_len]
        pitch_strength = pitch_strength[:min_len]
        vt_energy = vt_energy[:min_len]

        # Normalize features
        def normalize(x):
            if np.std(x) < 1e-10:
                return np.zeros_like(x)
            return (x - np.mean(x)) / (np.std(x) + 1e-10)

        zcr_norm = normalize(zcr)
        energy_norm = normalize(energy)
        pitch_norm = normalize(pitch_strength)
        vt_norm = normalize(vt_energy)

        # Decision rule: Voiced if:
        # - Low ZCR (< threshold)
        # - High energy (> threshold)
        # - Strong pitch periodicity (> threshold)
        # - Strong vocal tract envelope (> threshold)
        voiced_score = (
            -0.3 * zcr_norm +      # Lower ZCR = more voiced
            0.3 * energy_norm +    # Higher energy = more voiced
            0.25 * pitch_norm +    # Stronger pitch = more voiced
            0.15 * vt_norm         # Stronger vocal tract = more voiced
        )

        # Threshold for voiced/unvoiced
        threshold = 0.0
        labels = ['voiced' if s > threshold else 'unvoiced' for s in voiced_score]

        # Find boundaries (transitions)
        boundaries = []
        frame_times = np.arange(len(labels)) * self.hop_length / self.sample_rate

        for i in range(1, len(labels)):
            if labels[i] != labels[i-1]:
                boundaries.append((frame_times[i], labels[i]))

        return boundaries, labels, voiced_score, frame_times

    def plot_detection(self, signal, boundaries, labels, voiced_score, frame_times, sample_rate=16000,PROJECT_PATH=None):
        """Visualize voiced/unvoiced detection"""
        fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)

        # Original signal
        t = np.arange(len(signal)) / sample_rate
        axes[0].plot(t, signal, linewidth=0.5)
        axes[0].set_title('Original Signal')
        axes[0].set_ylabel('Amplitude')
        axes[0].grid(True, alpha=0.3)
        axes[0].set_xlim(0, min(3.0, t[-1]))  # Show first 3 seconds

        # Voiced score over time
        axes[1].plot(frame_times, voiced_score, linewidth=2, color='blue')
        axes[1].axhline(y=0, linestyle='--', color='red', label='Threshold')
        axes[1].set_title('Voiced Score (higher = more voiced)')
        axes[1].set_ylabel('Score')
        axes[1].legend()
        axes[1].grid(True, alpha=0.3)
        axes[1].set_xlim(0, min(3.0, frame_times[-1]))

        # Frame labels
        label_numeric = [1 if l == 'voiced' else 0 for l in labels]
        axes[2].plot(frame_times, label_numeric, drawstyle='steps-post', linewidth=2)
        axes[2].set_title('Detected Labels')
        axes[2].set_ylabel('Voiced (1) / Unvoiced (0)')
        axes[2].set_yticks([0, 1])
        axes[2].grid(True, alpha=0.3)
        axes[2].set_xlim(0, min(3.0, frame_times[-1]))

        # Boundary markers on signal
        axes[3].plot(t, signal, linewidth=0.5, alpha=0.5)
        for time, label in boundaries:
            if time < 3.0:  # Only show boundaries in visible range
                axes[3].axvline(x=time, color='red' if label == 'voiced' else 'blue',
                              linestyle='--', linewidth=1)
        axes[3].set_title('Detected Boundaries')
        axes[3].set_xlabel('Time (s)')
        axes[3].set_ylabel('Amplitude')
        axes[3].grid(True, alpha=0.3)
        axes[3].set_xlim(0, min(3.0, t[-1]))

        plt.tight_layout()
        plt.savefig(f"{PROJECT_PATH}/outputs/boundary_detection.pdf", dpi=300, bbox_inches='tight')
        plt.close()

        print(f" Boundary detection plot saved")
        print(f" Found {len(boundaries)} boundaries")
        print(f" Total frames: {len(labels)}")
        print(f" Voiced frames: {sum(1 for l in labels if l == 'voiced')}")
        print(f" Unvoiced frames: {sum(1 for l in labels if l == 'unvoiced')}")

        return boundaries

# Add this helper to voiced_unvoiced.py
def load_processed_signal(project_path):
    """Load signal saved by mfcc_manual.py"""
    signal_file = f"{project_path}/data/processed_signal.npy"
    if os.path.exists(signal_file):
        data = np.load(signal_file, allow_pickle=True).item()
        return data['signal'], data['sample_rate']
    return None, None


if __name__ == "__main__":
    print(" Detecting Voiced/Unvoiced Boundaries...")
    PROJECT_PATH = "/content/drive/MyDrive/q1_cepstral_pipeline"
    '''# Generate test signal with known voiced/unvoiced segments
    sample_rate = 16000
    duration = 3.0
    t = np.linspace(0, duration, int(sample_rate * duration))

    # Create segments: voiced-unvoiced-voiced
    signal = np.zeros_like(t)

    # 0-1s: Voiced (harmonic)
    signal[:sample_rate] = 0.5*np.sin(2*np.pi*150*t[:sample_rate]) + 0.2*np.sin(2*np.pi*300*t[:sample_rate])

    # 1-2s: Unvoiced (noise)
    signal[sample_rate:2*sample_rate] = 0.15*np.random.randn(sample_rate)

    # 2-3s: Voiced (different pitch)
    signal[2*sample_rate:] = 0.5*np.sin(2*np.pi*200*t[2*sample_rate:]) + 0.2*np.sin(2*np.pi*400*t[2*sample_rate:])

    # Add slight noise
    signal += 0.02 * np.random.randn(len(signal))'''
    import sys,os
    sys.path.insert(0, PROJECT_PATH)

    # Try to load processed signal first
    signal, sample_rate = load_processed_signal(PROJECT_PATH)

    if signal is None:
        # Fall back to synthetic
        print("⚠️  No processed signal found, generating synthetic...")
        # ... (synthetic generation code)
    else:
        print(f"✅ Using processed signal: {len(signal)/sample_rate:.2f}s @ {sample_rate}Hz")

    # Detect boundaries
    detector = VoicedUnvoicedDetector(sample_rate=sample_rate)
    boundaries, labels, scores, times = detector.detect_boundaries(signal)

    # Plot
    detector.plot_detection(signal, boundaries, labels, scores, times, sample_rate,PROJECT_PATH)

    # Print boundaries
    print("\n Detected Boundaries:")
    for time, label in boundaries:
        print(f"  {time:.2f}s → {label}")

    # Expected boundaries at ~1.0s and ~2.0s
    print("\n Expected: ~1.0s (voiced→unvoiced), ~2.0s (unvoiced→voiced)")

Overwriting /content/drive/MyDrive/q1_cepstral_pipeline/voiced_unvoiced.py


In [9]:
%%writefile $PROJECT_PATH/phonetic_mapping.py
"""
Phonetic Mapping using Hugging Face Wav2Vec2 for Forced Alignment
Supports: Real audio files (.wav, .m4a, .flac, .mp3)
"""

import numpy as np
import torch
from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor
from scipy.interpolate import interp1d
import matplotlib.pyplot as plt
import json
import os
import sys

DEFAULT_PROJECT_PATH = "/content/drive/MyDrive/q1_cepstral_pipeline"

def load_audio_for_alignment(filepath, target_sr=16000, max_duration=30):
    """
    Load and prepare audio for Wav2Vec2 alignment

    Args:
        filepath: Path to audio file
        target_sr: Target sample rate (Wav2Vec2 expects 16kHz)
        max_duration: Max duration in seconds to avoid OOM

    Returns:
        audio: Normalized mono audio as numpy array
        sample_rate: Actual sample rate
    """
    try:
        import torchaudio
        waveform, sr = torchaudio.load(filepath)
        if waveform.shape[0] > 1:
            waveform = waveform.mean(dim=0, keepdim=True)
        if sr != target_sr:
            waveform = torchaudio.functional.resample(waveform, sr, target_sr)
            sr = target_sr
        audio = waveform.numpy().flatten()
        audio = audio / (np.max(np.abs(audio)) + 1e-8)

        # Trim long files to avoid memory issues
        max_samples = int(max_duration * sr)
        if len(audio) > max_samples:
            print(f"  Trimming audio from {len(audio)/sr:.1f}s to {max_duration}s")
            audio = audio[:max_samples]

        return audio, sr
    except Exception as e:
        print(f" Error loading {filepath}: {e}")
        return None, None


class PhoneticAligner:
    """Use Wav2Vec2 for phonetic alignment and boundary comparison"""

    def __init__(self, model_name="facebook/wav2vec2-base-960h", device=None):
        """
        Args:
            model_name: HuggingFace model identifier
            device: 'cuda' or 'cpu' (auto-detected if None)
        """
        if device is None:
            device = 'cuda' if torch.cuda.is_available() else 'cpu'

        print(f" Loading {model_name} on {device}...")
        self.device = device
        self.processor = Wav2Vec2Processor.from_pretrained(model_name)
        self.model = Wav2Vec2ForCTC.from_pretrained(model_name).to(device)
        self.model.eval()
        print(f" Model loaded on {device}")

    def align_audio(self, audio, sample_rate=16000):
        """
        Get frame-level predictions from Wav2Vec2

        Returns:
            predictions: list of (time, token, confidence)
            transcription: decoded text string
        """
        # Ensure input is 16kHz mono
        if sample_rate != 16000:
            import torchaudio
            audio_tensor = torch.tensor(audio).unsqueeze(0)
            audio_tensor = torchaudio.functional.resample(audio_tensor, sample_rate, 16000)
            audio = audio_tensor.numpy().flatten()
            sample_rate = 16000

        # Preprocess for Wav2Vec2
        input_values = self.processor(audio, sampling_rate=sample_rate,
                                     return_tensors="pt").input_values.to(self.device)

        # Forward pass
        with torch.no_grad():
            logits = self.model(input_values).logits

        # Get predictions
        predicted_ids = torch.argmax(logits, dim=-1)[0].cpu()
        transcription = self.processor.batch_decode(predicted_ids)[0]

        # Extract frame-level info (~50 fps for Wav2Vec2)
        frame_rate = 50
        frame_duration = 1.0 / frame_rate

        predictions = []
        for i, token_id in enumerate(predicted_ids):
            token = self.processor.tokenizer.decode([token_id]).strip()
            if token:  # Skip blank tokens
                confidence = torch.softmax(logits[0, i], dim=0)[token_id].cpu().item()
                time = i * frame_duration
                predictions.append({
                    'time': time,
                    'token': token,
                    'confidence': confidence
                })

        return predictions, transcription

    def compute_boundary_rmse(self, manual_boundaries, model_predictions, tolerance=0.05):
        """
        Compute RMSE between manual and model boundaries

        Args:
            manual_boundaries: list of (time, label) from voiced_unvoiced.py
            model_predictions: list of (time, token, confidence) from align_audio
            tolerance: seconds within which to match boundaries

        Returns:
            dict with RMSE and matching statistics
        """
        # Extract model boundary candidates (token transitions)
        model_boundaries = []
        for i in range(1, len(model_predictions)):
            if model_predictions[i]['token'] != model_predictions[i-1]['token']:
                model_boundaries.append(model_predictions[i]['time'])

        if len(model_boundaries) == 0:
            return {
                'rmse_seconds': float('inf'),
                'rmse_ms': float('inf'),
                'matched_count': 0,
                'error': 'No model boundaries detected'
            }

        # Match manual to model boundaries
        matched = []
        unmatched_manual = []
        model_times = model_boundaries.copy()

        for man_time, man_label in manual_boundaries:
            if len(model_times) == 0:
                unmatched_manual.append((man_time, man_label))
                continue
            closest_idx = np.argmin(np.abs(np.array(model_times) - man_time))
            model_time = model_times[closest_idx]
            error = abs(man_time - model_time)

            if error <= tolerance:
                matched.append((man_time, model_time, error))
                model_times.pop(closest_idx)
            else:
                unmatched_manual.append((man_time, man_label))

        # Calculate RMSE
        if len(matched) > 0:
            errors = [e for _, _, e in matched]
            rmse = np.sqrt(np.mean(np.array(errors)**2))
        else:
            rmse = float('inf')

        return {
            'rmse_seconds': rmse,
            'rmse_ms': rmse * 1000,
            'matched_count': len(matched),
            'total_manual': len(manual_boundaries),
            'total_model': len(model_boundaries),
            'unmatched_manual': len(unmatched_manual),
            'matched_pairs': matched
        }

    def plot_alignment_comparison(self, audio, sample_rate, manual_boundaries,
                                 output_path, title="Alignment Comparison"):
        """Visualize manual vs model alignment"""
        predictions, transcription = self.align_audio(audio, sample_rate)
        results = self.compute_boundary_rmse(manual_boundaries, predictions)

        fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
        t = np.arange(len(audio)) / sample_rate

        # Original signal
        axes[0].plot(t, audio, linewidth=0.5)
        axes[0].set_title(f'Original Signal\nTranscription: "{transcription[:60]}{"..." if len(transcription)>60 else ""}"')
        axes[0].set_ylabel('Amplitude')
        axes[0].grid(True, alpha=0.3)
        axes[0].set_xlim(0, min(5.0, t[-1]))

        # Manual boundaries
        axes[1].plot(t, audio, linewidth=0.3, alpha=0.3)
        for time, label in manual_boundaries:
            if time < 5.0:
                axes[1].axvline(x=time, color='blue', linestyle='--', linewidth=2,
                              label=f'Manual: {label}' if time == manual_boundaries[0][0] else "")
        axes[1].set_title('Manual Boundaries (Voiced/Unvoiced)')
        axes[1].set_ylabel('Amplitude')
        axes[1].legend(fontsize=8, loc='upper right')
        axes[1].grid(True, alpha=0.3)
        axes[1].set_xlim(0, min(5.0, t[-1]))

        # Model boundaries
        axes[2].plot(t, audio, linewidth=0.3, alpha=0.3)
        model_times = [p['time'] for p in predictions if p['token']]
        for mt in model_times[::3]:  # Plot every 3rd to avoid clutter
            if mt < 5.0:
                axes[2].axvline(x=mt, color='red', linestyle=':', linewidth=1)
        axes[2].set_title('Model Token Boundaries (Wav2Vec2)')
        axes[2].set_xlabel('Time (s)')
        axes[2].set_ylabel('Amplitude')
        axes[2].grid(True, alpha=0.3)
        axes[2].set_xlim(0, min(5.0, t[-1]))

        # RMSE annotation
        rmse_text = f"RMSE: {results['rmse_ms']:.1f} ms" if results['rmse_ms'] != float('inf') else "RMSE: N/A"
        axes[0].text(0.02, 0.95, rmse_text,
                    transform=axes[0].transAxes,
                    bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5),
                    fontsize=10)

        plt.suptitle(title, fontsize=14, fontweight='bold', y=1.02)
        plt.tight_layout()
        plt.savefig(output_path, dpi=300, bbox_inches='tight')
        plt.close()

        return results


def run_full_pipeline(audio_path, project_path=None, skip_model=False):
    """
    Run complete phonetic mapping pipeline

    Args:
        audio_path: Path to audio file
        project_path: Output directory
        skip_model: If True, skip Wav2Vec2 (for testing)

    Returns:
        results dict or None
    """
    if project_path is None:
        project_path = DEFAULT_PROJECT_PATH

    print(f" Running Phonetic Mapping: {os.path.basename(audio_path)}")

    # Load audio
    audio, sample_rate = load_audio_for_alignment(audio_path, target_sr=16000, max_duration=30)
    if audio is None:
        return None

    print(f" Loaded: {len(audio)/sample_rate:.2f}s @ {sample_rate} Hz")

    if skip_model:
        print("  Skipping Wav2Vec2 (skip_model=True)")
        return {'skipped': True, 'duration': len(audio)/sample_rate}

    # Detect manual boundaries first
    from voiced_unvoiced import VoicedUnvoicedDetector
    detector = VoicedUnvoicedDetector(sample_rate=sample_rate)
    manual_boundaries, labels, scores, times = detector.detect_boundaries(audio)
    print(f" Detected {len(manual_boundaries)} manual boundaries")

    # Align with Wav2Vec2
    try:
        aligner = PhoneticAligner()
        output_path = f"{project_path}/outputs/alignment_results.pdf"
        results = aligner.plot_alignment_comparison(
            audio, sample_rate, manual_boundaries, output_path,
            title=f"Alignment: {os.path.basename(audio_path)}"
        )

        # Save results
        with open(f"{project_path}/outputs/alignment_results.json", 'w') as f:
            json.dump(results, f, indent=2)

        print("\n" + "="*60)
        print(" PHONETIC MAPPING RESULTS")
        print("="*60)
        print(f"RMSE: {results['rmse_ms']:.2f} ms" if results['rmse_ms'] != float('inf') else "RMSE: Could not compute")
        print(f"Matched boundaries: {results['matched_count']}/{results['total_manual']}")
        print(f"Unmatched manual: {results['unmatched_manual']}")
        print(f"Output: {output_path}")
        print("="*60)

        return results

    except Exception as e:
        print(f" Error during alignment: {e}")
        print(" Tip: Wav2Vec2 requires ~2GB RAM and ~30s for first load")
        return {'error': str(e)}


def run_demo_synthetic(project_path=None):
    """Demo with synthetic signal (no model download)"""
    if project_path is None:
        project_path = DEFAULT_PROJECT_PATH

    print(" Phonetic Mapping Demo (Synthetic - No Model Download)")

    # Create simple test signal
    sample_rate = 16000
    duration = 2.0
    t = np.linspace(0, duration, int(sample_rate * duration))
    audio = np.zeros_like(t)
    audio[:sample_rate//2] = 0.5*np.sin(2*np.pi*150*t[:sample_rate//2])
    audio[sample_rate//2:sample_rate] = 0.1*np.random.randn(sample_rate//2)
    audio[sample_rate:] = 0.5*np.sin(2*np.pi*200*t[sample_rate:])

    print("  Using synthetic audio for demo")
    print(" For real results: python phonetic_mapping.py --input your_file.m4a")

    # Show what would happen
    print("\n Expected workflow with real audio:")
    print("  1. Load audio → Resample to 16kHz")
    print("  2. Detect voiced/unvoiced boundaries (cepstral method)")
    print("  3. Run Wav2Vec2 for token-level alignment")
    print("  4. Compute RMSE between manual and model boundaries")
    print("  5. Generate visualization PDF")

    return {'demo': True}


if __name__ == "__main__":
    import argparse
    parser = argparse.ArgumentParser(description='Phonetic Mapping with Wav2Vec2')
    parser.add_argument('--input', '-i', type=str, default=None, help='Path to real audio file')
    parser.add_argument('--project-path', '-p', type=str, default=DEFAULT_PROJECT_PATH)
    parser.add_argument('--skip-model', action='store_true', help='Skip Wav2Vec2 download (demo mode)')
    parser.add_argument('--synthetic', '-s', action='store_true', help='Use synthetic demo')
    args = parser.parse_args()

    os.makedirs(f"{args.project_path}/outputs", exist_ok=True)

    if args.synthetic or (not args.input and not os.path.exists(args.input if args.input else "")):
        run_demo_synthetic(args.project_path)
    elif args.input and os.path.exists(args.input):
        run_full_pipeline(args.input, args.project_path, skip_model=args.skip_model)
    else:
        print("  No input file provided. Use --input path/to/audio.m4a")
        print(" Or use --synthetic for demo mode")

Overwriting /content/drive/MyDrive/q1_cepstral_pipeline/phonetic_mapping.py


In [20]:
%%writefile $PROJECT_PATH/phonetic_mapping_full.py
"""
Phonetic Mapping using Hugging Face Wav2Vec2 for Forced Alignment
Maps detected segments to actual phones (e.g., [p] vs [b]) and computes RMSE
"""

import numpy as np
import torch
from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor, Wav2Vec2CTCTokenizer
from scipy.interpolate import interp1d
import matplotlib.pyplot as plt
import json
import os
import sys
import re
from collections import defaultdict

DEFAULT_PROJECT_PATH = "/content/drive/MyDrive/q1_cepstral_pipeline"

# IPA phone mapping for CTC tokens (simplified)
CTC_TO_PHONE = {
    ' ': '<sil>', '<pad>': '<sil>', '<unk>': '<unk>',
    'a': 'ɑ', 'e': 'ɛ', 'i': 'ɪ', 'o': 'ɔ', 'u': 'ʊ',
    'aa': 'ɑ', 'ae': 'ɛ', 'ah': 'ʌ', 'ao': 'ɔ', 'aw': 'aʊ',
    'ay': 'aɪ', 'b': 'b', 'ch': 'tʃ', 'd': 'd', 'dh': 'ð',
    'eh': 'ɛ', 'er': 'ɝ', 'ey': 'eɪ', 'f': 'f', 'g': 'ɡ',
    'hh': 'h', 'ih': 'ɪ', 'iy': 'i', 'jh': 'dʒ', 'k': 'k',
    'l': 'l', 'm': 'm', 'n': 'n', 'ng': 'ŋ', 'ow': 'oʊ',
    'oy': 'ɔɪ', 'p': 'p', 'r': 'ɹ', 's': 's', 'sh': 'ʃ',
    't': 't', 'th': 'θ', 'uh': 'ʊ', 'uw': 'u', 'v': 'v',
    'w': 'w', 'y': 'j', 'z': 'z', 'zh': 'ʒ',
}

def load_audio_for_alignment(filepath, target_sr=16000, max_duration=30):
    """Load and prepare audio for Wav2Vec2 alignment"""
    try:
        import torchaudio
        waveform, sr = torchaudio.load(filepath)
        if waveform.shape[0] > 1:
            waveform = waveform.mean(dim=0, keepdim=True)
        if sr != target_sr:
            waveform = torchaudio.functional.resample(waveform, sr, target_sr)
            sr = target_sr
        audio = waveform.numpy().flatten()
        audio = audio / (np.max(np.abs(audio)) + 1e-8)

        # Trim long files
        max_samples = int(max_duration * sr)
        if len(audio) > max_samples:
            print(f"⚠️  Trimming audio from {len(audio)/sr:.1f}s to {max_duration}s")
            audio = audio[:max_samples]

        return audio, sr
    except Exception as e:
        print(f"❌ Error loading {filepath}: {e}")
        return None, None


class PhoneticAligner:
    """Wav2Vec2-based phonetic alignment with phone-level mapping"""

    def __init__(self, model_name="facebook/wav2vec2-base-960h", device=None):
        if device is None:
            device = 'cuda' if torch.cuda.is_available() else 'cpu'

        print(f"🔄 Loading {model_name} on {device}...")
        self.device = device
        self.processor = Wav2Vec2Processor.from_pretrained(model_name)
        self.model = Wav2Vec2ForCTC.from_pretrained(model_name).to(device)
        self.tokenizer = Wav2Vec2CTCTokenizer.from_pretrained(model_name)
        self.model.eval()
        self.frame_rate = 50  # Wav2Vec2 outputs ~50 frames/sec
        print(f"✅ Model loaded on {device} (frame_rate={self.frame_rate}fps)")

    def _decode_tokens(self, token_ids):
        """Convert token IDs to phone sequence with timing"""
        phones = []
        for token_id in token_ids:
            token = self.tokenizer.decode([token_id]).strip().lower()
            if token and token not in ['<pad>', '<s>', '</s>', '<unk>']:
                # Map CTC token to IPA phone
                phone = CTC_TO_PHONE.get(token, token)
                phones.append(phone)
        return phones

    def align_audio(self, audio, sample_rate=16000):
        """
        Get phone-level alignment with timestamps

        Returns:
            phone_alignments: list of {phone, start_time, end_time, confidence}
            transcription: decoded text
        """
        # Ensure 16kHz
        if sample_rate != 16000:
            import torchaudio
            audio_tensor = torch.tensor(audio).unsqueeze(0)
            audio_tensor = torchaudio.functional.resample(audio_tensor, sample_rate, 16000)
            audio = audio_tensor.numpy().flatten()
            sample_rate = 16000

        # Preprocess
        input_values = self.processor(audio, sampling_rate=sample_rate,
                                     return_tensors="pt").input_values.to(self.device)

        # Forward pass
        with torch.no_grad():
            logits = self.model(input_values).logits
            probabilities = torch.softmax(logits, dim=-1)

        # Greedy decoding
        predicted_ids = torch.argmax(logits, dim=-1)[0].cpu()
        transcription = self.processor.batch_decode(predicted_ids)[0]

        # Extract phone-level alignments with timing
        phone_alignments = []
        current_phone = None
        start_frame = 0

        for i, token_id in enumerate(predicted_ids):
            token = self.tokenizer.decode([token_id]).strip().lower()
            if not token or token in ['<pad>', '<s>', '</s>', '<unk>']:
                continue

            phone = CTC_TO_PHONE.get(token, token)
            confidence = probabilities[0, i, token_id].cpu().item()

            if phone != current_phone:
                # Save previous phone segment
                if current_phone is not None:
                    phone_alignments.append({
                        'phone': current_phone,
                        'start_time': start_frame / self.frame_rate,
                        'end_time': i / self.frame_rate,
                        'confidence': np.mean([p['confidence'] for p in phone_alignments
                                              if p['phone'] == current_phone]) if phone_alignments else confidence
                    })
                # Start new phone
                current_phone = phone
                start_frame = i

        # Don't forget the last phone
        if current_phone is not None:
            phone_alignments.append({
                'phone': current_phone,
                'start_time': start_frame / self.frame_rate,
                'end_time': len(predicted_ids) / self.frame_rate,
                'confidence': confidence
            })

        return phone_alignments, transcription

    def extract_phone_boundaries(self, phone_alignments):
        """Extract boundary times from phone alignments"""
        boundaries = []
        for p in phone_alignments:
            # Mark phone start as boundary
            boundaries.append((p['start_time'], f"phone:{p['phone']}"))
        return boundaries

    def compute_boundary_rmse(self, manual_boundaries, phone_boundaries, tolerance=0.1):
        """
        Compute RMSE between manual voiced/unvoiced boundaries and phone boundaries

        Args:
            manual_boundaries: list of (time, label) from voiced_unvoiced.py
            phone_boundaries: list of (time, phone_label) from Wav2Vec2
            tolerance: seconds within which to consider a match

        Returns:
            dict with RMSE and detailed matching stats
        """
        if len(phone_boundaries) == 0:
            return {
                'rmse_seconds': float('inf'),
                'rmse_ms': float('inf'),
                'matched_count': 0,
                'error': 'No phone boundaries detected'
            }

        # Extract just times for matching
        manual_times = [t for t, _ in manual_boundaries]
        phone_times = [t for t, _ in phone_boundaries]

        # Match each manual boundary to nearest phone boundary
        matched = []
        unmatched_manual = []
        used_phone_idx = set()

        for man_time, man_label in manual_boundaries:
            # Find closest unused phone boundary
            best_error = float('inf')
            best_idx = None

            for idx, phone_time in enumerate(phone_times):
                if idx in used_phone_idx:
                    continue
                error = abs(man_time - phone_time)
                if error < best_error:
                    best_error = error
                    best_idx = idx

            if best_idx is not None and best_error <= tolerance:
                matched.append({
                    'manual_time': man_time,
                    'manual_label': man_label,
                    'phone_time': phone_times[best_idx],
                    'phone_label': phone_boundaries[best_idx][1],
                    'error': best_error
                })
                used_phone_idx.add(best_idx)
            else:
                unmatched_manual.append((man_time, man_label))

        # Calculate RMSE
        if len(matched) > 0:
            errors = [m['error'] for m in matched]
            rmse = np.sqrt(np.mean(np.array(errors)**2))
        else:
            rmse = float('inf')

        return {
            'rmse_seconds': rmse,
            'rmse_ms': rmse * 1000 if rmse != float('inf') else float('inf'),
            'matched_count': len(matched),
            'total_manual': len(manual_boundaries),
            'total_phone': len(phone_boundaries),
            'unmatched_manual': len(unmatched_manual),
            'tolerance_seconds': tolerance,
            'matched_details': matched
        }

    def plot_phone_alignment(self, audio, sample_rate, manual_boundaries, phone_alignments,
                            output_path, title="Phone-Level Alignment"):
        """Visualize phone-level alignment with manual boundaries"""
        fig, axes = plt.subplots(4, 1, figsize=(16, 12), sharex=True)
        t = np.arange(len(audio)) / sample_rate

        # 1. Original waveform
        axes[0].plot(t, audio, linewidth=0.5, color='gray')
        axes[0].set_title('Original Audio Waveform')
        axes[0].set_ylabel('Amplitude')
        axes[0].grid(True, alpha=0.3)
        axes[0].set_xlim(0, min(10.0, t[-1]))

        # 2. Manual voiced/unvoiced boundaries
        axes[1].plot(t, audio, linewidth=0.3, alpha=0.2)
        for time, label in manual_boundaries:
            if time < 10.0:
                color = 'blue' if label == 'voiced' else 'red'
                axes[1].axvline(x=time, color=color, linestyle='--', linewidth=2,
                              label=label if time == manual_boundaries[0][0] else "")
        axes[1].set_title('Manual Boundaries: Voiced (blue) / Unvoiced (red)')
        axes[1].set_ylabel('Amplitude')
        axes[1].legend(fontsize=8)
        axes[1].grid(True, alpha=0.3)
        axes[1].set_xlim(0, min(10.0, t[-1]))

        # 3. Phone-level alignment (colored bars)
        axes[2].plot(t, audio, linewidth=0.1, alpha=0.1)
        colors = plt.cm.tab20(np.linspace(0, 1, 20))
        phone_color_map = {p: colors[i % len(colors)] for i, p in enumerate(set(p['phone'] for p in phone_alignments))}

        for p in phone_alignments:
            if p['start_time'] < 10.0:
                axes[2].axvspan(p['start_time'], p['end_time'],
                               color=phone_color_map.get(p['phone'], 'gray'),
                               alpha=0.3, label=p['phone'] if p == phone_alignments[0] else "")
                # Phone label
                mid_time = (p['start_time'] + p['end_time']) / 2
                if mid_time < 10.0:
                    axes[2].text(mid_time, 0.3, p['phone'], fontsize=7,
                                rotation=90, ha='center', va='bottom', alpha=0.8)

        axes[2].set_title('Wav2Vec2 Phone-Level Alignment')
        axes[2].set_ylabel('Amplitude')
        axes[2].grid(True, alpha=0.3)
        axes[2].set_xlim(0, min(10.0, t[-1]))

        # 4. Boundary comparison (matched vs unmatched)
        axes[3].plot(t, audio, linewidth=0.1, alpha=0.1)

        # Plot matched boundaries (green)
        for m in [d for d in phone_alignments if any(d['phone'] in str(b) for b in manual_boundaries)][:10]:
            if m['start_time'] < 10.0:
                axes[3].axvline(x=m['start_time'], color='green', linestyle='-', linewidth=2, alpha=0.7)

        # Plot manual boundaries (blue/red)
        for time, label in manual_boundaries:
            if time < 10.0:
                color = 'blue' if label == 'voiced' else 'red'
                axes[3].axvline(x=time, color=color, linestyle='--', linewidth=1.5)

        axes[3].set_title('Boundary Comparison: Manual (dashed) vs Phone (solid green=matched)')
        axes[3].set_xlabel('Time (seconds)')
        axes[3].set_ylabel('Amplitude')
        axes[3].grid(True, alpha=0.3)
        axes[3].set_xlim(0, min(10.0, t[-1]))

        plt.suptitle(title, fontsize=14, fontweight='bold', y=1.02)
        plt.tight_layout()
        plt.savefig(output_path, dpi=300, bbox_inches='tight')
        plt.close()

        print(f" Alignment plot saved: {output_path}")
        return output_path

    def generate_phone_report(self, phone_alignments, manual_boundaries, rmse_results, output_path):
        """Generate detailed phone-level report"""
        report = []
        report.append("# Phone-Level Alignment Report\n")
        report.append(f"**Audio Duration**: {phone_alignments[-1]['end_time']:.2f}s\n")
        report.append(f"**Total Phones Detected**: {len(phone_alignments)}\n")
        report.append(f"**Manual Boundaries**: {len(manual_boundaries)}\n\n")

        # Phone sequence
        report.append("## Detected Phone Sequence\n")
        report.append("| Time (s) | Phone | Duration (ms) | Confidence |\n")
        report.append("|----------|-------|---------------|------------|\n")
        for p in phone_alignments[:20]:  # First 20 phones
            duration = (p['end_time'] - p['start_time']) * 1000
            report.append(f"| {p['start_time']:.3f} | `{p['phone']}` | {duration:.1f} | {p['confidence']:.3f} |\n")
        if len(phone_alignments) > 20:
            report.append(f"\n*... and {len(phone_alignments) - 20} more phones*\n")

        # RMSE results
        report.append("\n## Boundary Alignment Results\n")
        if rmse_results['rmse_ms'] != float('inf'):
            report.append(f"- **RMSE**: {rmse_results['rmse_ms']:.2f} ms\n")
            report.append(f"- **Matched Boundaries**: {rmse_results['matched_count']}/{rmse_results['total_manual']}\n")
            report.append(f"- **Tolerance**: {rmse_results['tolerance_seconds']*1000:.0f} ms\n")
        else:
            report.append("- **RMSE**: Could not compute (no matches)\n")

        # Matched pairs detail
        if rmse_results['matched_details']:
            report.append("\n### Matched Boundary Pairs\n")
            report.append("| Manual Time | Manual Label | Phone Time | Phone | Error (ms) |\n")
            report.append("|-------------|--------------|------------|-------|------------|\n")
            for m in rmse_results['matched_details'][:10]:
                report.append(f"| {m['manual_time']:.3f}s | {m['manual_label']} | {m['phone_time']:.3f}s | {m['phone_label']} | {m['error']*1000:.1f} |\n")

        # Save report
        with open(output_path, 'w') as f:
            f.write(''.join(report))

        print(f" Phone report saved: {output_path}")
        return output_path


def run_full_pipeline(audio_path, project_path=None):
    """
    Run complete phonetic mapping pipeline with phone-level alignment

    Returns:
        results dict with RMSE and alignment details
    """
    if project_path is None:
        project_path = DEFAULT_PROJECT_PATH

    os.makedirs(f"{project_path}/outputs", exist_ok=True)

    print(f" Running Phonetic Mapping: {os.path.basename(audio_path)}")

    # Load audio
    audio, sample_rate = load_audio_for_alignment(audio_path, target_sr=16000, max_duration=30)
    if audio is None:
        return None

    print(f" Loaded: {len(audio)/sample_rate:.2f}s @ {sample_rate} Hz")

    # Detect manual boundaries first
    print(" Detecting manual voiced/unvoiced boundaries...")
    from voiced_unvoiced import VoicedUnvoicedDetector
    detector = VoicedUnvoicedDetector(sample_rate=sample_rate)
    manual_boundaries, labels, scores, times = detector.detect_boundaries(audio)
    print(f" Detected {len(manual_boundaries)} manual boundaries")

    # Initialize aligner and run alignment
    print(" Loading Wav2Vec2 model...")
    aligner = PhoneticAligner()

    print(" Running phone-level alignment...")
    phone_alignments, transcription = aligner.align_audio(audio, sample_rate)
    print(f" Detected {len(phone_alignments)} phones: {' '.join(p['phone'] for p in phone_alignments[:15])}...")

    # Extract phone boundaries for RMSE calculation
    phone_boundaries = aligner.extract_phone_boundaries(phone_alignments)

    # Compute RMSE
    print(" Computing boundary RMSE...")
    rmse_results = aligner.compute_boundary_rmse(manual_boundaries, phone_boundaries, tolerance=0.1)

    # Generate outputs
    plot_path = f"{project_path}/outputs/alignment_results.pdf"
    aligner.plot_phone_alignment(audio, sample_rate, manual_boundaries, phone_alignments,
                                 plot_path, title=f"Phone Alignment: {os.path.basename(audio_path)}")

    report_path = f"{project_path}/outputs/phone_report.md"
    aligner.generate_phone_report(phone_alignments, manual_boundaries, rmse_results, report_path)

    # Save JSON results
    json_path = f"{project_path}/outputs/alignment_results.json"
    with open(json_path, 'w') as f:
        json.dump({
            'transcription': transcription,
            'phone_count': len(phone_alignments),
            'manual_boundary_count': len(manual_boundaries),
            'rmse_results': rmse_results,
            'first_10_phones': phone_alignments[:10]
        }, f, indent=2)

    # Print summary
    print("\n" + "="*70)
    print(" PHONETIC MAPPING RESULTS")
    print("="*70)
    print(f"Transcription: \"{transcription}\"")
    print(f"Phones detected: {len(phone_alignments)}")
    print(f"Manual boundaries: {len(manual_boundaries)}")
    if rmse_results['rmse_ms'] != float('inf'):
        print(f" RMSE: {rmse_results['rmse_ms']:.2f} ms")
        print(f" Matched: {rmse_results['matched_count']}/{rmse_results['total_manual']} boundaries")
    else:
        print(f"  RMSE: Could not compute (tolerance={rmse_results['tolerance_seconds']*1000:.0f}ms)")
    print(f" Plot: {plot_path}")
    print(f" Report: {report_path}")
    print("="*70)

    return {
        'transcription': transcription,
        'phones': phone_alignments,
        'manual_boundaries': manual_boundaries,
        'rmse_results': rmse_results,
        'outputs': {
            'plot': plot_path,
            'report': report_path,
            'json': json_path
        }
    }


def run_demo_synthetic(project_path=None):
    """Demo with synthetic signal (no model download)"""
    if project_path is None:
        project_path = DEFAULT_PROJECT_PATH

    print(" Phonetic Mapping Demo (Synthetic - No Model)")
    print("\n Expected workflow with real audio:")
    print("  1. Load audio → Resample to 16kHz")
    print("  2. Detect voiced/unvoiced boundaries (cepstral method)")
    print("  3. Run Wav2Vec2 for phone-level alignment")
    print("  4. Map CTC tokens to IPA phones ([p], [b], [t], etc.)")
    print("  5. Compute RMSE between manual and phone boundaries")
    print("  6. Generate visualization + report")
    print("\n For real results: run with --input your_file.m4a")

    return {'demo': True}


if __name__ == "__main__":
    import argparse
    parser = argparse.ArgumentParser(description='Phonetic Mapping with Wav2Vec2')
    parser.add_argument('--input', '-i', type=str, default=None, help='Path to real audio file')
    parser.add_argument('--project-path', '-p', type=str, default=DEFAULT_PROJECT_PATH)
    parser.add_argument('--synthetic', '-s', action='store_true', help='Use synthetic demo')

    #  FIX: Use parse_known_args() to ignore Jupyter's hidden arguments
    args, unknown = parser.parse_known_args()

    # Optional: Print unknown args for debugging
    # if unknown:
    #     print(f"  Ignoring unknown arguments: {unknown}")

    if args.synthetic or (not args.input):
        run_demo_synthetic(args.project_path)
    elif args.input and os.path.exists(args.input):
        results = run_full_pipeline(args.input, args.project_path)
        if results and results['rmse_results']['rmse_ms'] != float('inf'):
            print(f"\n Final RMSE for submission: {results['rmse_results']['rmse_ms']:.2f} ms")
    else:
        print(f" File not found: {args.input}")
        print(" Usage: python phonetic_mapping.py --input /path/to/audio.m4a")

Overwriting /content/drive/MyDrive/q1_cepstral_pipeline/phonetic_mapping_full.py


In [21]:
# 🎤 RUN FULL PHONETIC MAPPING (WITH WAV2VEC2)
PROJECT_PATH = "/content/drive/MyDrive/q1_cepstral_pipeline"
AUDIO_FILE = f"{PROJECT_PATH}/data/voice_speech_Q1_1.m4a"

print("="*70)
print(" RUNNING FULL PHONETIC MAPPING WITH WAV2VEC2")
print("="*70)

# Verify setup
import torch
print(f" GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only'}")
print(f" Audio file: {AUDIO_FILE}")
print(f" Exists: {os.path.exists(AUDIO_FILE)}")

# Run full pipeline (this will download Wav2Vec2 ~350MB)
print("\n Starting phonetic mapping (first run: ~2 minutes)...")
%run $PROJECT_PATH/phonetic_mapping_full.py --input "$AUDIO_FILE" --project-path "$PROJECT_PATH"

print("\n Phonetic mapping complete!")

🎯 RUNNING FULL PHONETIC MAPPING WITH WAV2VEC2
✅ GPU: CPU only
✅ Audio file: /content/drive/MyDrive/q1_cepstral_pipeline/data/voice_speech_Q1_1.m4a
✅ Exists: True

🚀 Starting phonetic mapping (first run: ~2 minutes)...
🔤 Running Phonetic Mapping: voice_speech_Q1_1.m4a
⚠️  Trimming audio from 32.6s to 30s
✅ Loaded: 30.00s @ 16000 Hz
📊 Detecting manual voiced/unvoiced boundaries...
✅ Detected 279 manual boundaries
🔄 Loading Wav2Vec2 model...
🔄 Loading facebook/wav2vec2-base-960h on cpu...


Loading weights:   0%|          | 0/212 [00:00<?, ?it/s]

Wav2Vec2ForCTC LOAD REPORT from: facebook/wav2vec2-base-960h
Key                        | Status  | 
---------------------------+---------+-
wav2vec2.masked_spec_embed | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✅ Model loaded on cpu (frame_rate=50fps)
🔤 Running phone-level alignment...
✅ Detected 224 phones: ɑ l ɔ n ɛ t ɔ t ɛ t ɪ s ʊ p ɛ...
📏 Computing boundary RMSE...
✅ Alignment plot saved: /content/drive/MyDrive/q1_cepstral_pipeline/outputs/alignment_results.pdf
✅ Phone report saved: /content/drive/MyDrive/q1_cepstral_pipeline/outputs/phone_report.md

📊 PHONETIC MAPPING RESULTS
Transcription: "<pad>"
Phones detected: 224
Manual boundaries: 279
✅ RMSE: 57.21 ms
✅ Matched: 141/279 boundaries
📄 Plot: /content/drive/MyDrive/q1_cepstral_pipeline/outputs/alignment_results.pdf
📄 Report: /content/drive/MyDrive/q1_cepstral_pipeline/outputs/phone_report.md

🎯 Final RMSE for submission: 57.21 ms

✅ Phonetic mapping complete!


<Figure size 640x480 with 0 Axes>

In [22]:
# 🎤 TEST BOTH MODULES WITH YOUR REAL FILE
PROJECT_PATH = "/content/drive/MyDrive/q1_cepstral_pipeline"
AUDIO_FILE = f"{PROJECT_PATH}/data/voice_speech_Q1_1.m4a"

print("="*70)
print(" TESTING REAL AUDIO SUPPORT")
print("="*70)

# Verify file exists
if not os.path.exists(AUDIO_FILE):
    print(f" File not found: {AUDIO_FILE}")
    print(" Upload voice_speech_Q1_1.m4a to your Google Drive folder")
else:
    print(f" Found: {AUDIO_FILE}")
    print(f" Size: {os.path.getsize(AUDIO_FILE)/1024:.1f} KB")

# Test 1: Spectral Leakage with real audio
print("\n Test 1: Spectral Leakage Analysis...")
%run $PROJECT_PATH/leakage_snr.py --input "$AUDIO_FILE" --project-path "$PROJECT_PATH"

# 3. Voiced/Unvoiced Detection
print("\n Step 3: Boundary Detection...")
%run $PROJECT_PATH/voiced_unvoiced.py

# Test 2: Phonetic Mapping (demo mode - skip model download)
print("\n Test 2: Phonetic Mapping (Demo Mode)...")
%run $PROJECT_PATH/phonetic_mapping.py --input "$AUDIO_FILE" --project-path "$PROJECT_PATH"

print("\n" + "="*70)
print(" TESTS COMPLETE")
print("="*70)

 TESTING REAL AUDIO SUPPORT
 Found: /content/drive/MyDrive/q1_cepstral_pipeline/data/voice_speech_Q1_1.m4a
 Size: 815.0 KB

 Test 1: Spectral Leakage Analysis...
 Analyzing Spectral Leakage & SNR for: voice_speech_Q1_1.m4a
 Analyzing 100.0ms segment @ 16000 Hz

 SPECTRAL LEAKAGE & SNR COMPARISON
Window            Leakage Ratio     Main Lobe (Hz)     SNR (dB)
----------------------------------------------------------------------
Rectangular               43.73              30.00         5.65
Hamming                   52.27              40.00         8.95
Hanning                   54.49              40.00        10.14

 Comparison plot saved to /content/drive/MyDrive/q1_cepstral_pipeline/outputs/leakage_comparison.pdf

 For this speech segment: 'Hanning' window recommended

 Step 3: Boundary Detection...
 Detecting Voiced/Unvoiced Boundaries...
✅ Using processed signal: 32.64s @ 16000Hz
 Boundary detection plot saved
 Found 294 boundaries
 Total frames: 3262
 Voiced frames: 1542
 Unvoice

Loading weights:   0%|          | 0/212 [00:00<?, ?it/s]

Wav2Vec2ForCTC LOAD REPORT from: facebook/wav2vec2-base-960h
Key                        | Status  | 
---------------------------+---------+-
wav2vec2.masked_spec_embed | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


 Model loaded on cpu

 PHONETIC MAPPING RESULTS
RMSE: 27.22 ms
Matched boundaries: 100/279
Unmatched manual: 179
Output: /content/drive/MyDrive/q1_cepstral_pipeline/outputs/alignment_results.pdf

 TESTS COMPLETE


<Figure size 640x480 with 0 Axes>

In [ ]:
#  CREATE SUBMISSION ZIP
import shutil
from google.colab import files

shutil.make_archive("/content/q1_submission", 'zip', PROJECT_PATH)
files.download("/content/q1_submission.zip")
print(" Submission package ready!")